# Salary Estimate Classification

This educational project uses job-posting features to predict whether an estimated annual salary is below or at least the median salary of the training data. This is a relative salary category, not a measure of employee satisfaction or an estimate of an individual's actual pay.

## 1. Load the data

Place `df.csv` beside this notebook.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv('df.csv')
print('Original shape:', raw.shape)
display(raw.head())
display(raw.isna().sum())

Original shape: (5892, 11)


,Job Title,Salary Estimate,Rating,Company Name,Location,Size,Founded,Type of ownership,Industry,Sector,Revenue
0,Senior Data Scientist,$111K-$181K (Glassdoor est.),3.5,Hopper\n3.5,"New York, NY",501 to 1000 employees,2007,Company - Private,Travel Agencies,Travel & Tourism,Unknown / Non-Applicable
1,"Data Scientist, Product Analytics",$111K-$181K (Glassdoor est.),4.5,Noom US\n4.5,"New York, NY",1001 to 5000 employees,2008,Company - Private,"Health, Beauty, & Fitness",Consumer Services,Unknown / Non-Applicable
2,Data Science Manager,$111K-$181K (Glassdoor est.),-1.0,Decode_M,"New York, NY",1 to 50 employees,-1,Unknown,-1,-1,Unknown / Non-Applicable
3,Data Analyst,$111K-$181K (Glassdoor est.),3.4,Sapphire Digital\n3.4,"Lyndhurst, NJ",201 to 500 employees,2019,Company - Private,Internet,Information Technology,Unknown / Non-Applicable
4,"Director, Data Science",$111K-$181K (Glassdoor est.),3.4,United Entertainment Group\n3.4,"New York, NY",51 to 200 employees,2007,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable


Job Title            0
Salary Estimate      0
Rating               0
Company Name         0
Location             0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
dtype: int64

## 2. Prepare salary estimates and features

Remove exact duplicates, unknown or hourly salaries, and records without a usable annual salary range. Extract annual salary ranges reported in thousands of dollars and use the midpoint as an approximation, not an actual salary. Drop rows with missing ratings or unusable location codes. Job titles are grouped using simple keyword rules. These are fixed transformations, not learned from the test set.

Excluding missing or hourly entries changes the population being analyzed. A two-letter location suffix is only a proxy for a US state and is not validated against a state list.

In [2]:
data = raw.drop_duplicates().copy()
salary_text = data['Salary Estimate'].fillna('').astype(str).str.strip()
# Only ranges explicitly marked in thousands (K) are interpreted as annual amounts.
annual = ~salary_text.str.contains('per hour|hourly', case=False, regex=True)
ranges = salary_text.str.extract(r'\$?\s*(\d+(?:\.\d+)?)\s*[kK]\s*-\s*\$?\s*(\d+(?:\.\d+)?)\s*[kK]')
minimum = pd.to_numeric(ranges[0], errors='coerce')
maximum = pd.to_numeric(ranges[1], errors='coerce')
data['Average_salary'] = (minimum + maximum) / 2
data['State'] = data['Location'].fillna('').str.split(',').str[-1].str.strip()
valid = (annual & minimum.notna() & maximum.notna() & (minimum <= maximum)
         & data['Rating'].notna() & (data['Rating'] != -1) & data['State'].str.len().eq(2))
data = data.loc[valid].copy()

def categorize_job_title(value):
    title = str(value).lower()
    if 'data scientist' in title or 'data science' in title: return 'data scientist'
    if 'data engineer' in title: return 'data engineer'
    if 'deep learning' in title: return 'deep learning engineer'
    if 'machine learning' in title: return 'machine learning engineer'
    if 'director' in title: return 'director'
    if 'analyst' in title: return 'analyst'
    if 'research' in title: return 'researcher'
    if 'manager' in title: return 'manager'
    return 'other'

data['Job Category'] = data['Job Title'].apply(categorize_job_title)
X = data[['Job Category', 'State', 'Rating']].copy()
salary = data['Average_salary'].copy()
print('Rows after cleaning:', len(data))
display(data[['Job Title', 'Job Category', 'State', 'Rating', 'Average_salary']].head())
print('Salary midpoint summary (thousands of dollars):')
print(salary.describe())

Rows after cleaning: 4859


,Job Title,Job Category,State,Rating,Average_salary
0,Senior Data Scientist,data scientist,NY,3.5,146.0
1,"Data Scientist, Product Analytics",data scientist,NY,4.5,146.0
3,Data Analyst,analyst,NJ,3.4,146.0
4,"Director, Data Science",data scientist,NY,3.4,146.0
5,Data Scientist,data scientist,NY,2.9,146.0


Salary midpoint summary (thousands of dollars):
count    4859.000000
mean      106.038485
std        35.811839
min        37.500000
25%        76.750000
50%       101.500000
75%       130.000000
max       268.000000
Name: Average_salary, dtype: float64


## 3. Reserve the test set and define the target

Split first, then calculate the threshold from training salaries only. Apply that same threshold to both partitions. The split is not stratified because the classification labels are created after the split.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, salary_train, salary_test = train_test_split(
    X, salary, test_size=0.30, random_state=101
)
threshold = salary_train.median()
y_train = (salary_train >= threshold).astype(int)
y_test = (salary_test >= threshold).astype(int)
print(f'Training median: ${threshold:,.1f}K')
print('Class 0: below the median; class 1: at or above the median')
print('Training labels:', y_train.value_counts().sort_index().to_dict())
print('Test labels:', y_test.value_counts().sort_index().to_dict())

Training median: $101.5K
Class 0: below the median; class 1: at or above the median
Training labels: {0: 1672, 1: 1729}
Test labels: {0: 724, 1: 734}


## 4. Compare models on training data

One-hot encoding and rating scaling are fitted inside each cross-validation fold. Unknown job categories or locations are ignored at prediction time. We compare logistic regression, K-nearest neighbors, decision tree and random forest with a majority-class baseline. Cross-validation uses labels defined by a single median from the full training partition, so results are conditional on that threshold.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

def build_pipeline(classifier):
    preprocessing = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore'), ['Job Category', 'State']),
        ('rating', StandardScaler(), ['Rating'])
    ])
    return Pipeline([('preprocess', preprocessing), ('classifier', classifier)])

models = {
    'Majority baseline': build_pipeline(DummyClassifier(strategy='most_frequent')),
    'Logistic regression': build_pipeline(LogisticRegression(max_iter=1000, random_state=101)),
    'K-nearest neighbors': build_pipeline(KNeighborsClassifier()),
    'Decision tree': build_pipeline(DecisionTreeClassifier(random_state=101, min_samples_leaf=5)),
    'Random forest': build_pipeline(RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=101, n_jobs=1))
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)
results = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring={'accuracy':'accuracy', 'f1':'f1'}, error_score='raise', n_jobs=1)
    results.append({'Model': name, 'CV accuracy': scores['test_accuracy'].mean(),
                    'CV accuracy std': scores['test_accuracy'].std(), 'CV F1': scores['test_f1'].mean()})
comparison = pd.DataFrame(results).sort_values('CV accuracy', ascending=False).reset_index(drop=True)
display(comparison.round(3))

,Model,CV accuracy,CV accuracy std,CV F1
0,Random forest,0.700,0.016,0.692
1,Logistic regression,0.696,0.011,0.689
2,Decision tree,0.674,0.019,0.670
3,K-nearest neighbors,0.666,0.013,0.671
4,Majority baseline,0.508,0.001,0.674


## 5. Evaluate the selected model on the test set

Select among the four predictive models by mean training cross-validation accuracy, fit the selected pipeline on all training data, and evaluate it on the test set. The baseline remains visible for comparison; if it performs better, that limitation is important.

In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

selected_name = comparison.loc[comparison['Model'] != 'Majority baseline', 'Model'].iloc[0]
selected_model = models[selected_name]
selected_model.fit(X_train, y_train)
predictions = selected_model.predict(X_test)
print('Selected model:', selected_name)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.3f}')
print('Confusion matrix (rows: actual; columns: predicted):')
print(confusion_matrix(y_test, predictions, labels=[0, 1]))
print(classification_report(y_test, predictions, target_names=['Below median', 'At or above median'], zero_division=0))

Selected model: Random forest
Test accuracy: 0.698
Confusion matrix (rows: actual; columns: predicted):
[[523 201]
 [239 495]]
                    precision    recall  f1-score   support

      Below median       0.69      0.72      0.70       724
At or above median       0.71      0.67      0.69       734

          accuracy                           0.70      1458
         macro avg       0.70      0.70      0.70      1458
      weighted avg       0.70      0.70      0.70      1458



## Limitations

- Salary estimates are ranges rather than confirmed compensation and the class is relative to the training sample.
- Filtering missing ratings, hourly estimates and nonstandard locations may introduce selection bias. Salary parsing assumes a range explicitly expressed in thousands.
- Some postings or companies may recur across the random split, so the test may not represent new employers or future postings. A company-grouped or time-based split would answer different questions.
- Choosing a model after cross-validation and reporting one held-out test split does not guarantee future performance. Future development should use a new evaluation strategy.
- This is an educational example, not advice for compensation decisions. Confirm data provenance, redistribution permission and time period before publication.